# 12F-08 — FuelCons modelling and pacification

**Question:** How do RUN results become a comparable FuelCons record?

In [ ]:
from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in search_roots if (path / "AGENTS.md").exists() and (path / "etl").exists())
import sqlite3

CANONICAL_DB = REPO_ROOT / "etl/data/staging/sprint_12e2_epa_fuelcons/eco_drive_canonical_epa_fuelcons.db"
connection = sqlite3.connect(CANONICAL_DB.resolve().as_uri() + "?mode=ro", uri=True)


## Load EPA two-cycle FuelCons records

In [ ]:
query = '''
SELECT id AS fuelcons_id, vde_id, electrification, fuel_type,
       fuel_ftp75_l_per_100km AS city_l_100km,
       fuel_hwfet_l_per_100km AS highway_l_100km,
       fuel_l_per_100km AS canonical_combined_l_100km,
       comparison_basis, provenance_json
FROM fuelcons
WHERE record_origin='EPA_RECONSTRUCTED'
  AND comparison_basis='EPA_LABEL_2_CYCLE'
  AND fuel_ftp75_l_per_100km IS NOT NULL
  AND fuel_hwfet_l_per_100km IS NOT NULL
'''
fuelcons = pd.read_sql_query(query, connection)
fuelcons.shape


## Reproduce the approved 55/45 combination

In [ ]:
fuelcons["notebook_combined_l_100km"] = 0.55 * fuelcons["city_l_100km"] + 0.45 * fuelcons["highway_l_100km"]
fuelcons["difference_l_100km"] = fuelcons["notebook_combined_l_100km"] - fuelcons["canonical_combined_l_100km"]
fuelcons[["city_l_100km", "highway_l_100km", "canonical_combined_l_100km", "difference_l_100km"]].head()


In [ ]:
assert fuelcons["difference_l_100km"].abs().max() < 1e-9
fuelcons.groupby("electrification")["canonical_combined_l_100km"].describe()


## Multi-RUN lineage remains visible

In [ ]:
adoption = pd.read_sql_query('''
SELECT fuelcons_id, COUNT(*) AS adopted_runs
FROM fuelcons_run_adoption
GROUP BY fuelcons_id
''', connection)
adoption["adopted_runs"].value_counts().sort_index()


In [ ]:
fuelcons.merge(adoption, on="fuelcons_id").sort_values("adopted_runs", ascending=False).head(10)


In [ ]:
connection.close()


## What did we learn?

FuelCons is a comparison result with explicit basis and adopted RUN lineage. A convertible number is not enough: conflicting evidence remains unresolved instead of being averaged opportunistically.